# 准备工作

In [2]:
# 检查 Python 环境

import sys
from pathlib import Path

import polars as pl
import pandas as pd
import pyarrow

print("Python:", sys.version)
print("Interpreter:", sys.executable)
print("Polars:", pl.__version__)
print("Pandas:", pd.__version__)
print("PyArrow:", pyarrow.__version__)

Python: 3.12.7 (tags/v3.12.7:0b05ead, Oct  1 2024, 03:06:41) [MSC v.1941 64 bit (AMD64)]
Interpreter: c:\Users\71907\Documents\Work\Project\.venv\Scripts\python.exe
Polars: 1.43.1
Pandas: 3.0.5
PyArrow: 25.0.0


In [3]:
# 设置文件路径

from pathlib import Path

PROJECT_DIR = Path.cwd().parent
RAW_DIR = PROJECT_DIR / "raw_data"
CLEAN_DIR = PROJECT_DIR / "cleaned_data"

CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)
print("Cleaned data directory:", CLEAN_DIR)

Project directory: c:\Users\71907\Documents\Work\Project
Raw data directory: c:\Users\71907\Documents\Work\Project\raw_data
Cleaned data directory: c:\Users\71907\Documents\Work\Project\cleaned_data


In [4]:
# 检查 raw_data 里有哪些文件

csv_files = sorted(RAW_DIR.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files:\n")

for file in csv_files:
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.name:<30} {size_mb:>10.2f} MB")

Found 7 CSV files:

assessments.csv                      0.01 MB
courses.csv                          0.00 MB
studentAssessment.csv                5.43 MB
studentInfo.csv                      3.30 MB
studentRegistration.csv              1.06 MB
studentVle.csv                     432.81 MB
vle.csv                              0.25 MB


In [5]:
# Create file path dictionary

FILES = {
    "courses": RAW_DIR / "courses.csv",
    "assessments": RAW_DIR / "assessments.csv",
    "vle": RAW_DIR / "vle.csv",
    "student_info": RAW_DIR / "studentInfo.csv",
    "student_registration": RAW_DIR / "studentRegistration.csv",
    "student_assessment": RAW_DIR / "studentAssessment.csv",
    "student_vle": RAW_DIR / "studentVle.csv",
}

for name, path in FILES.items():
    print(f"{name:<25} exists={path.exists()}  path={path.name}")

courses                   exists=True  path=courses.csv
assessments               exists=True  path=assessments.csv
vle                       exists=True  path=vle.csv
student_info              exists=True  path=studentInfo.csv
student_registration      exists=True  path=studentRegistration.csv
student_assessment        exists=True  path=studentAssessment.csv
student_vle               exists=True  path=studentVle.csv


# 开始分批处理数据

In [6]:
# 读取六个较小的数据文件

courses = pl.read_csv(
    FILES["courses"],
    null_values=["", "NA", "?", "null"]
)

assessments = pl.read_csv(
    FILES["assessments"],
    null_values=["", "NA", "?", "null"]
)

vle = pl.read_csv(
    FILES["vle"],
    null_values=["", "NA", "?", "null"]
)

student_info = pl.read_csv(
    FILES["student_info"],
    null_values=["", "NA", "?", "null"]
)

student_registration = pl.read_csv(
    FILES["student_registration"],
    null_values=["", "NA", "?", "null"]
)

student_assessment = pl.read_csv(
    FILES["student_assessment"],
    null_values=["", "NA", "?", "null"]
)

print("All six smaller datasets loaded successfully.")

All six smaller datasets loaded successfully.


In [7]:
# 查看数据大小

datasets = {
    "courses": courses,
    "assessments": assessments,
    "vle": vle,
    "student_info": student_info,
    "student_registration": student_registration,
    "student_assessment": student_assessment,
}

for name, df in datasets.items():
    print(f"{name:<25} rows={df.height:<10} columns={df.width}")

courses                   rows=22         columns=3
assessments               rows=206        columns=6
vle                       rows=6364       columns=6
student_info              rows=32593      columns=12
student_registration      rows=32593      columns=5
student_assessment        rows=173912     columns=5


In [8]:
# 查看列名和 data types

for name, df in datasets.items():
    print(f"\n{'=' * 60}")
    print(name)
    print(df.schema)


courses
Schema({'code_module': String, 'code_presentation': String, 'module_presentation_length': Int64})

assessments
Schema({'code_module': String, 'code_presentation': String, 'id_assessment': Int64, 'assessment_type': String, 'date': Int64, 'weight': Float64})

vle
Schema({'id_site': Int64, 'code_module': String, 'code_presentation': String, 'activity_type': String, 'week_from': String, 'week_to': String})

student_info
Schema({'code_module': String, 'code_presentation': String, 'id_student': Int64, 'gender': String, 'region': String, 'highest_education': String, 'imd_band': String, 'age_band': String, 'num_of_prev_attempts': Int64, 'studied_credits': Int64, 'disability': String, 'final_result': String})

student_registration
Schema({'code_module': String, 'code_presentation': String, 'id_student': Int64, 'date_registration': Int64, 'date_unregistration': Int64})

student_assessment
Schema({'id_assessment': Int64, 'id_student': Int64, 'date_submitted': Int64, 'is_banked': Int64, '

这个 output 大部分其实是正常的，不是全部有问题。

真正需要处理的是：

vle.week_from → String 和
vle.week_to   → String

它们理论上应该是 numerical columns，但因为里面有大量空值，Polars 自动把它们识别成了 String。

其他这些都正常：

date → Int64 / 
score → Int64 / 
weight → Float64 / 
id_student → Int64

In [9]:
vle.select(["week_from", "week_to"]).head(20)

week_from,week_to
str,str
null,null
null,null
null,null
null,null
null,null
…,…
null,null
null,null
null,null


这说明 week_from 和 week_to 现在已经是 i64，也就是 integer，不是 String。

刚才最开始贴的 schema 里显示 String，可能是后来重新读取或转换后，当前 notebook 里的 vle 已经更新了。现在不需要再转换

In [10]:
# 检查这两列到底有哪些非空值

vle.filter(
    pl.col("week_from").is_not_null()
).select(
    ["week_from", "week_to"]
).head(20)

week_from,week_to
str,str
"""2""","""2"""
"""1""","""1"""
"""1""","""1"""
"""2""","""2"""
"""1""","""1"""
…,…
"""14""","""14"""
"""4""","""4"""
"""25""","""25"""


output 正常：

week_from 和 week_to 都是 integer
非空值看起来合理
这 20 行里两列是成对出现的

这里没有问题，可以继续

## 检查 missing values

判断： missing value 是 error、structural missingness，还是需要 indicator / imputation

### vle

In [11]:
# 检查两列的 missing count

vle.select(
    pl.col("week_from").null_count().alias("week_from_missing"),
    pl.col("week_to").null_count().alias("week_to_missing")
)

week_from_missing,week_to_missing
u32,u32
5243,5243


两列缺失数完全一致，说明它们大概率是成对缺失，不是单独哪一列坏掉。

In [12]:
# 检查是否存在“一列有值、另一列缺失”的情况

vle.filter(
    pl.col("week_from").is_null() != pl.col("week_to").is_null()
).shape

(0, 6)

(0, 6) 表示：

没有任何 row 出现 week_from 缺失但 week_to 有值
也没有 week_to 缺失但 week_from 有值
两列始终一起缺失、一起出现

所以这里不用 cleaning，也不用填补。week_from 和 week_to 先原样保留。

In [13]:
# 检查 assessments$date 的缺失情况

assessments.filter(
    pl.col("date").is_null()
)

code_module,code_presentation,id_assessment,assessment_type,date,weight
str,str,i64,str,i64,f64
"""AAA""","""2013J""",1757,"""Exam""",null,100.0
"""AAA""","""2014J""",1763,"""Exam""",null,100.0
"""BBB""","""2013B""",14990,"""Exam""",null,100.0
"""BBB""","""2013J""",15002,"""Exam""",null,100.0
"""BBB""","""2014B""",15014,"""Exam""",null,100.0
…,…,…,…,…,…
"""CCC""","""2014B""",24290,"""Exam""",null,100.0
"""CCC""","""2014B""",40087,"""Exam""",null,100.0
"""CCC""","""2014J""",24299,"""Exam""",null,100.0


这个 output 是正常的：11 个 missing date 全部来自 assessment_type = "Exam"，而且 weight = 100.0。

这属于 structural missingness，不是普通 data error。这里不要：
drop these rows / 
impute date / 
fill with mean or median / 
fill with course end date

因为在 OULAD 里，部分 Exam 的 exact assessment date 本来就没有记录。

In [14]:
# 确认是不是所有 missing date 都是 Exam

assessments.filter(
    pl.col("date").is_null()
).group_by(
    "assessment_type"
).len()

assessment_type,len
str,u32
"""Exam""",11


确认了：所有 missing date 都来自 Exam。

所以这一项的 cleaning decision 是：

Keep the rows +
Keep date as null +
Do not impute

### student_assessments

In [15]:
# 检查 student_assessment.score 的 missing rows 数量

student_assessment.filter(
    pl.col("score").is_null()
).shape

(173, 5)

In [16]:
# 看看这些 missing score 属于哪些 assessment_type

(
    student_assessment
    .filter(pl.col("score").is_null())
    .join(
        assessments.select(
            ["id_assessment", "assessment_type"]
        ),
        on="id_assessment",
        how="left"
    )
    .group_by("assessment_type")
    .len()
)

assessment_type,len
str,u32
"""TMA""",173


In [17]:
# 只检查这些 rows 的 date_submitted 和 is_banked

student_assessment.filter(
    pl.col("score").is_null()
).select(
    ["id_assessment", "id_student", "date_submitted", "is_banked", "score"]
).head(20)

id_assessment,id_student,date_submitted,is_banked,score
i64,i64,i64,i64,i64
1752,721259,22,0,null
1754,260355,127,0,null
1760,2606802,180,0,null
14984,186780,77,0,null
14984,531205,26,0,null
…,…,…,…,…
14988,554818,158,0,null
14989,478317,222,0,null
14989,502717,216,0,null


In [18]:
# 统计 missing score 中 is_banked 的分布

student_assessment.filter(
    pl.col("score").is_null()
).group_by(
    "is_banked"
).len()

is_banked,len
i64,u32
1,1
0,172


In [19]:
# 检查这 173 rows 的 date_submitted 分布，特别是有没有异常负值

student_assessment.filter(
    pl.col("score").is_null()
).select(
    pl.col("date_submitted").min().alias("min_date_submitted"),
    pl.col("date_submitted").max().alias("max_date_submitted"),
    (pl.col("date_submitted") < 0).sum().alias("negative_date_count")
)

min_date_submitted,max_date_submitted,negative_date_count
i64,i64,u32
-2,274,2


这个结果说明，在 173 个 missing score rows 中：

date_submitted range 是 -2 到 274

只有 2 rows 的 date_submitted < 0

negative date 不一定是错误，因为 OULAD 的 date 是相对于 course start 的时间，负数代表 course start 前提交

In [20]:
# 只看这 2 rows

student_assessment.filter(
    pl.col("score").is_null() &
    (pl.col("date_submitted") < 0)
)

id_assessment,id_student,date_submitted,is_banked,score
i64,i64,i64,i64,i64
14997,501208,-1,1,null
25334,555297,-2,0,null


这两条也不一定是错误：

date_submitted = -1, is_banked = 1：很像 banked assessment record

date_submitted = -2, is_banked = 0：可能是 course start 前提交

目前的 cleaning decision 仍然是：

Keep both rows +
Keep score as null +
Do not convert negative dates to 0

In [21]:
# 检查这两个 id_assessment 对应的 assessment information

assessments.filter(
    pl.col("id_assessment").is_in([14997, 25334])
)

code_module,code_presentation,id_assessment,assessment_type,date,weight
str,str,i64,str,i64,f64
"""BBB""","""2013J""",14997,"""TMA""",47,18.0
"""DDD""","""2013B""",25334,"""TMA""",25,7.5


这两个都是正常的 TMA，assessment deadlines 分别是 day 47 和 day 25。负的 date_submitted 表示在 course start 前提交，并不矛盾

In [22]:
# 检查这 173 个 missing score 是否集中在少数 assessments

(
    student_assessment
    .filter(pl.col("score").is_null())
    .group_by("id_assessment")
    .len()
    .sort("len", descending=True)
    .head(20)
)

id_assessment,len
i64,u32
15013,7
25339,7
34860,6
25363,5
14984,5
…,…
14989,3
34902,3
34890,3


这个结果说明 missing score 分散在多个 assessments 中，不是某一个 assessment 整列损坏；单个 assessment 最多只有 7 条

In [23]:
# 检查这些 missing score rows 对应的 final_result 分布

(
    student_assessment
    .filter(pl.col("score").is_null())
    .join(
        assessments.select(
            ["id_assessment", "code_module", "code_presentation"]
        ),
        on="id_assessment",
        how="left"
    )
    .join(
        student_info.select(
            [
                "code_module",
                "code_presentation",
                "id_student",
                "final_result"
            ]
        ),
        on=["code_module", "code_presentation", "id_student"],
        how="left"
    )
    .group_by("final_result")
    .len()
    .sort("len", descending=True)
)

final_result,len
str,u32
"""Withdrawn""",72
"""Fail""",67
"""Pass""",34


这个结果很有信息：

Withdrawn: 72 + 
Fail: 67 + 
Pass: 34 + 
没有 Distinction

说明 missing score 更集中在 lower-performing / withdrawn students，而不是完全随机缺失。

所以这里不能简单当作普通 missing data 处理，否则可能丢掉重要的 behavioral signal。

In [24]:
# 给 student_assessment 增加一个 missing indicator，但先不填补 score

student_assessment = student_assessment.with_columns(
    pl.col("score")
    .is_null()
    .cast(pl.Int8)
    .alias("score_missing")
)

In [25]:
student_assessment.select(
    ["score", "score_missing"]
).head(10)

score,score_missing
i64,i8
78,0
70,0
72,0
69,0
79,0
70,0
72,0
72,0
71,0


正常，score_missing 已经成功建立：

有 score 的 rows → score_missing = 0

missing score 的 rows → 应该是 score_missing = 1

In [26]:
student_assessment.group_by(
    "score_missing"
).len().sort("score_missing")

score_missing,len
i8,u32
0,173739
1,173


数量和前面的检查一致，说明这个 indicator 建立成功。

现在 score 这一项先告一段落，cleaning decision 是：

Keep missing scores as null  +
Keep all rows +
Add score_missing indicator +
Do not impute with 0

### student_info

In [27]:
# 检查一个新问题：student_info.imd_band 的 missing values

student_info.select(
    pl.col("imd_band").null_count().alias("imd_band_missing")
)

imd_band_missing
u32
1111


In [28]:
# 查看 imd_band 的所有 category 和数量

student_info.group_by(
    "imd_band"
).len().sort(
    "imd_band"
)

imd_band,len
str,u32
null,1111
"""0-10%""",3311
"""10-20""",3516
"""20-30%""",3654
"""30-40%""",3539
…,…
"""50-60%""",3124
"""60-70%""",2905
"""70-80%""",2879


imd_band 有 1111 missing values

"10-20" 只是命名少了 %，与其他 categories 不一致

In [29]:
# 统一 category naming

student_info = student_info.with_columns(
    pl.when(pl.col("imd_band") == "10-20")
    .then(pl.lit("10-20%"))
    .otherwise(pl.col("imd_band"))
    .alias("imd_band")
)

In [30]:
# 把 missing values 设为独立 category

student_info = student_info.with_columns(
    pl.col("imd_band").fill_null("Unknown")
)

student_info.group_by(
    "imd_band"
).len().sort(
    "imd_band"
)

imd_band,len
str,u32
"""0-10%""",3311
"""10-20%""",3516
"""20-30%""",3654
"""30-40%""",3539
"""40-50%""",3256
…,…
"""60-70%""",2905
"""70-80%""",2879
"""80-90%""",2762


这一步处理正确：

category naming 已统一为 "10-20%"

1111 个 missing values 已转换成 "Unknown"

imd_band 现在没有 null

In [31]:
# 确认 missing count 已经变成 0

student_info.select(
    pl.col("imd_band").null_count().alias("imd_band_missing")
)

imd_band_missing
u32
0


当前 cleaning decision：

Standardize "10-20" to "10-20%" +
Replace missing imd_band with "Unknown" + 
No rows removed

### student_registeration

In [32]:
# 检查 student_registration.date_registration 的 missing values

student_registration.select(
    pl.col("date_registration").null_count().alias("date_registration_missing")
)

date_registration_missing
u32
45


date_registration 有 45 missing values。

这类 missing value 不能马上用 0 或 mean 填补，因为 date_registration 是相对于 course start 的 registration date，0 本身有实际含义

In [33]:
# 把这 45 rows 看出来

student_registration.filter(
    pl.col("date_registration").is_null()
).head(20)

code_module,code_presentation,id_student,date_registration,date_unregistration
str,str,i64,i64,i64
"""BBB""","""2013B""",630346,null,null
"""BBB""","""2013J""",57369,null,-1
"""BBB""","""2013J""",342678,null,-33
"""BBB""","""2014B""",582496,null,-126
"""BBB""","""2014B""",607646,null,-38
…,…,…,…,…
"""CCC""","""2014J""",680333,null,-65
"""CCC""","""2014J""",1777834,null,null
"""DDD""","""2013B""",128965,null,-24


In [34]:
# 只检查这 45 rows 的 date_unregistration 情况

student_registration.filter(
    pl.col("date_registration").is_null()
).select(
    pl.col("date_unregistration").null_count().alias("unregistration_missing"),
    pl.col("date_unregistration").is_not_null().sum().alias("unregistration_available")
)

unregistration_missing,unregistration_available
u32,u32
6,39


在 45 个 missing date_registration rows 中：

39 rows 有 date_unregistration + 6 rows 连 date_unregistration 也缺失

这类情况不能简单填 0，因为 0 代表 course start 当天 registration，不是“未知”

In [36]:
# 给 date_registration 增加一个 missing indicator，原始值继续保留为 null

student_registration = student_registration.with_columns(
    pl.col("date_registration")
    .is_null()
    .cast(pl.Int8)
    .alias("registration_date_missing")
)

student_registration.group_by(
    "registration_date_missing"
).len().sort(
    "registration_date_missing"
)

registration_date_missing,len
i8,u32
0,32548
1,45


In [37]:
# 只检查 date_unregistration 的 missing count

student_registration.select(
    pl.col("date_unregistration").null_count().alias("date_unregistration_missing")
)

date_unregistration_missing
u32
22521


数量很大，但它主要是 structural missingness：

date_unregistration = null 通常表示学生no withdrawal
不能把这些值填成 0

也不能删除这些 rows

下一步只做一件事：建立一个 indicator，表示学生是否有 unregistration record

In [38]:
# 建立一个 indicator，表示学生是否有 unregistration record

student_registration = student_registration.with_columns(
    pl.col("date_unregistration")
    .is_not_null()
    .cast(pl.Int8)
    .alias("has_unregistered")
)

student_registration.group_by(
    "has_unregistered"
).len().sort(
    "has_unregistered"
)

has_unregistered,len
i8,u32
0,22521
1,10072


重点：has_unregistered cannot be used directly as a predictor at early cutoffs

因为它可能包含 Week 2 / Week 4 之后才发生的 information，会造成 temporal leakage。它可以保留在 cleaned dataset 里，但后面建模时要按 cutoff 重新定义。

In [39]:
# 检查有没有 date_unregistration 早于 date_registration 的明显异常

student_registration.filter(
    pl.col("date_registration").is_not_null() &
    pl.col("date_unregistration").is_not_null() &
    (pl.col("date_unregistration") < pl.col("date_registration"))
).shape

(0, 7)

有任何 row 出现：

date_unregistration < date_registration

所以 registration chronology 没有发现明显异常。

目前 student_registration 这部分已经处理完：

Keep missing date_registration as null + 
Add registration_date_missing + 
Keep missing date_unregistration as structural missingness + 
Add has_unregistered + 
No impossible date order found

## 检查 key duplicate 

In [40]:
# 检查一个新问题：full duplicate rows

student_registration.height - student_registration.unique().height

0

0 表示 student_registration 里没有 full duplicate rows。


In [ ]:
# 只检查 key duplicates，因为同一条记录未必整行完全相同，但理论上下面这个 key 应该唯一:
# code_module + code_presentation + id_student

(
    student_registration
    .group_by(
        ["code_module", "code_presentation", "id_student"]
    )
    .len()
    .filter(pl.col("len") > 1)
)

code_module,code_presentation,id_student,len
str,str,i64,u32


新的 output 只有 column headers，没有 data rows，表示结果是一个 empty DataFrame。也就是：

No duplicate composite keys

因此：

code_module + code_presentation + id_student

在 student_registration 中是唯一的，符合它作为 one student–module–presentation registration record 的结构。


In [43]:
# 检查 student_info 的 composite key duplicates

(
    student_info
    .group_by(
        ["code_module", "code_presentation", "id_student"]
    )
    .len()
    .filter(pl.col("len") > 1)
)

code_module,code_presentation,id_student,len
str,str,i64,u32


In [ ]:
# 只检查 student_assessment 的 key duplicates

(
    student_assessment
    .group_by(
        ["id_assessment", "id_student"]
    )
    .len()
    .filter(pl.col("len") > 1)
)

id_assessment,id_student,len
i64,i64,u32


也是 empty DataFrame，说明：

id_assessment + id_student

在 student_assessment 中没有 duplicate keys。也就是说，同一个学生对同一个 assessment 只有一条记录，符合 data description。

目前 duplicate check 的结论是：

student_registration: no duplicate composite keys

student_info: no duplicate composite keys

student_assessment: no duplicate composite keys

In [45]:
# 只检查 courses 的 key

(
    courses
    .group_by(
        ["code_module", "code_presentation"]
    )
    .len()
    .filter(pl.col("len") > 1)
)

code_module,code_presentation,len
str,str,u32


这也是 empty DataFrame，说明：

code_module + code_presentation

在 courses 中是唯一的，没有 duplicate keys

In [46]:
# 只检查 assessments 的 id_assessment 是否唯一

(
    assessments
    .group_by("id_assessment")
    .len()
    .filter(pl.col("len") > 1)
)

id_assessment,len
i64,u32


也是 empty DataFrame，说明 id_assessment 在 assessments 中唯一，没有 duplicate keys

In [47]:
# 只检查 vle.id_site 是否唯一

(
    vle
    .group_by("id_site")
    .len()
    .filter(pl.col("len") > 1)
)

id_site,len
i64,u32


这也是 empty DataFrame，说明 id_site 在 vle 中唯一，没有 duplicate keys

到这里，六个较小 datasets 的 key duplicate check 都通过了

## 检查 unusual data

In [48]:
# 检查 score 是否都在 data description 规定的 0–100 range 内

student_assessment.filter(
    pl.col("score").is_not_null() &
    (
        (pl.col("score") < 0) |
        (pl.col("score") > 100)
    )
)

id_assessment,id_student,date_submitted,is_banked,score,score_missing
i64,i64,i64,i64,i64,i8


这也是 empty DataFrame，说明所有非缺失 score 都在 0–100 range 内，符合 data description。

所以下一步不需要修改 score values

In [49]:
# 检查 weight 是否存在 negative values

assessments.filter(
    pl.col("weight") < 0
)

code_module,code_presentation,id_assessment,assessment_type,date,weight
str,str,i64,str,i64,f64


这也是 empty DataFrame，说明没有 negative weight

In [50]:
# 检查 weight 有没有超过 100

assessments.filter(
    pl.col("weight") > 100
)

code_module,code_presentation,id_assessment,assessment_type,date,weight
str,str,i64,str,i64,f64


这也是 empty DataFrame，说明所有 weight <= 100

In [51]:
# 检查一个更重要的问题：同一个 code_module + code_presentation 下，non-Exam assessments 的 weight 是否合计为 100。
# 因为根据 description，Exam 是 separately calculated

(
    assessments
    .filter(pl.col("assessment_type") != "Exam")
    .group_by(["code_module", "code_presentation"])
    .agg(
        pl.col("weight").sum().alias("total_non_exam_weight")
    )
    .sort(["code_module", "code_presentation"])
)

code_module,code_presentation,total_non_exam_weight
str,str,f64
"""AAA""","""2013J""",100.0
"""AAA""","""2014J""",100.0
"""BBB""","""2013B""",100.0
"""BBB""","""2013J""",100.0
"""BBB""","""2014B""",100.0
…,…,…
"""FFF""","""2014B""",100.0
"""FFF""","""2014J""",100.0
"""GGG""","""2013J""",0.0


AAA 到 FFF 的 non-Exam weight 都是 100

GGG 是 0

根据 description，这不一定是 error。更可能是 GGG 的 assessment structure 不同，例如它的 assessments 可能全部是 CMA 但 weight 记为 0，或者 final result 的计算方式不同。

现在先不要改 GGG

In [52]:
# 检查 GGG 的 assessment records

assessments.filter(
    pl.col("code_module") == "GGG"
).sort(
    ["code_presentation", "date"]
)

code_module,code_presentation,id_assessment,assessment_type,date,weight
str,str,i64,str,i64,f64
"""GGG""","""2013J""",37415,"""TMA""",61,0.0
"""GGG""","""2013J""",37416,"""TMA""",124,0.0
"""GGG""","""2013J""",37417,"""TMA""",173,0.0
"""GGG""","""2013J""",37418,"""CMA""",229,0.0
"""GGG""","""2013J""",37419,"""CMA""",229,0.0
…,…,…,…,…,…
"""GGG""","""2014J""",37440,"""CMA""",229,0.0
"""GGG""","""2014J""",37441,"""CMA""",229,0.0
"""GGG""","""2014J""",37442,"""CMA""",229,0.0


结果说明 GGG 不是 data error，而是 assessment design 本身不同：

所有 TMA / CMA 的 weight = 0 + 
Exam 的 weight = 100

因此 non-Exam weight total = 0 是合理的

所以这里不需要修改 weight

In [53]:
# 检查每个 code_module + code_presentation 的 Exam weight 是否都是 100

(
    assessments
    .filter(pl.col("assessment_type") == "Exam")
    .group_by(["code_module", "code_presentation"])
    .agg(
        pl.col("weight").sum().alias("total_exam_weight")
    )
    .sort(["code_module", "code_presentation"])
)

code_module,code_presentation,total_exam_weight
str,str,f64
"""AAA""","""2013J""",100.0
"""AAA""","""2014J""",100.0
"""BBB""","""2013B""",100.0
"""BBB""","""2013J""",100.0
"""BBB""","""2014B""",100.0
…,…,…
"""FFF""","""2014B""",100.0
"""FFF""","""2014J""",100.0
"""GGG""","""2013J""",100.0


这个 output 正常：

每个 code_module + code_presentation 的 Exam weight 都是 100

assessment weight structure 与 description 一致

不需要修改任何 weight

In [54]:
# 检查 is_banked 是否只有合法值 0 和 1

student_assessment.group_by(
    "is_banked"
).len().sort(
    "is_banked"
)

is_banked,len
i64,u32
0,172003
1,1909


结果正常：

is_banked = 0: 172,003 rows

is_banked = 1: 1,909 rows

没有其他 unexpected values

所以 is_banked 不需要 cleaning

In [55]:
# 检查 date_submitted 是否存在明显超过 module length 的情况。
# 先把 student_assessment 连接到 assessments 和 courses

submitted_date_check = (
    student_assessment
    .join(
        assessments.select(
            [
                "id_assessment",
                "code_module",
                "code_presentation"
            ]
        ),
        on="id_assessment",
        how="left"
    )
    .join(
        courses,
        on=["code_module", "code_presentation"],
        how="left"
    )
)

submitted_date_check.filter(
    pl.col("date_submitted") > pl.col("module_presentation_length")
).shape

(85, 9)

有 85 条 date_submitted 超过了 module_presentation_length。

这不一定是 data error，可能是：

late submission / 
resubmission / 
banked assessment / 
assessment date 接近或超过 module end

现在先不要删除。

In [56]:
# 只看这些 rows 的基本情况

(
    submitted_date_check
    .filter(
        pl.col("date_submitted") >
        pl.col("module_presentation_length")
    )
    .select(
        [
            "id_assessment",
            "id_student",
            "date_submitted",
            "is_banked",
            "score",
            "code_module",
            "code_presentation",
            "module_presentation_length"
        ]
    )
    .sort(
        "date_submitted",
        descending=True
    )
    .head(20)
)

id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,module_presentation_length
i64,i64,i64,i64,i64,str,str,i64
34878,325750,608,0,74,"""FFF""","""2013J""",268
34879,325750,608,0,95,"""FFF""","""2013J""",268
34880,325750,608,0,68,"""FFF""","""2013J""",268
34881,325750,608,0,66,"""FFF""","""2013J""",268
34882,325750,608,0,68,"""FFF""","""2013J""",268
…,…,…,…,…,…,…,…
34879,572213,591,0,89,"""FFF""","""2013J""",268
34893,628476,591,0,76,"""FFF""","""2014B""",241
34894,628476,591,0,78,"""FFF""","""2014B""",241


这些不是普通的 late submissions。date_submitted 达到 590–608，而 module length 只有 241–268，而且同一学生在多个 assessments 上同时出现相同的超大日期，更像是 data anomaly 或特殊 administrative record，不能直接当正常 submission 使用。

先不删

In [57]:
# 检查这 85 rows 是否集中在少数学生

(
    submitted_date_check
    .filter(
        pl.col("date_submitted") >
        pl.col("module_presentation_length")
    )
    .group_by(
        ["code_module", "code_presentation", "id_student"]
    )
    .len()
    .sort("len", descending=True)
    .head(20)
)

code_module,code_presentation,id_student,len
str,str,i64,u32
"""FFF""","""2013J""",325750,5
"""FFF""","""2014B""",628476,4
"""FFF""","""2013B""",547884,4
"""FFF""","""2013B""",549792,3
"""FFF""","""2013B""",77998,3
…,…,…,…
"""FFF""","""2013J""",571844,2
"""FFF""","""2014B""",583056,2
"""FFF""","""2014B""",136944,1


这个 output 说明这 85 rows 主要集中在 FFF，而且每个 student 常常有多条 assessment records 同时出现超大 date_submitted。这更像 FFF 里的特殊 resubmission / administrative pattern，不建议直接删

In [58]:
# 看看这 85 rows 的 is_banked 分布

(
    submitted_date_check
    .filter(
        pl.col("date_submitted") >
        pl.col("module_presentation_length")
    )
    .group_by("is_banked")
    .len()
    .sort("is_banked")
)

is_banked,len
i64,u32
0,85


这说明 85 条都不是 banked assessment，所以不能用 is_banked = 1 来解释。

目前更像是 FFF 中少量 students 的特殊 late submission / repeated submission records。先不删除，因为删掉可能损失真实 behavior

In [59]:
# 看这些 rows 对应的 final_result 分布

(
    submitted_date_check
    .filter(
        pl.col("date_submitted") >
        pl.col("module_presentation_length")
    )
    .join(
        student_info.select(
            [
                "code_module",
                "code_presentation",
                "id_student",
                "final_result"
            ]
        ),
        on=["code_module", "code_presentation", "id_student"],
        how="left"
    )
    .group_by("final_result")
    .len()
    .sort("len", descending=True)
)

final_result,len
str,u32
"""Withdrawn""",70
"""Pass""",10
"""Fail""",4
"""Distinction""",1


85 rows 里大部分是 Withdrawn，说明它们更可能是特殊的 administrative / late-recording pattern，而不是随机错误。

先保留，但后面做 early prediction 时不会让这些 late records 进入 cutoff 之前的 features

In [60]:
# 这些异常 rows 的 date_submitted 比对应 assessment date 晚多少

(
    submitted_date_check
    .filter(
        pl.col("date_submitted") >
        pl.col("module_presentation_length")
    )
    .join(
        assessments.select(
            [
                "id_assessment",
                "date"
            ]
        ),
        on="id_assessment",
        how="left",
        suffix="_assessment"
    )
    .with_columns(
        (
            pl.col("date_submitted") -
            pl.col("date")
        ).alias("submission_delay")
    )
    .select(
        [
            "id_assessment",
            "id_student",
            "date_submitted",
            "date",
            "submission_delay",
            "code_module",
            "code_presentation"
        ]
    )
    .sort(
        "submission_delay",
        descending=True
    )
    .head(20)
)

id_assessment,id_student,date_submitted,date,submission_delay,code_module,code_presentation
i64,i64,i64,i64,i64,str,str
24290,577245,243,null,null,"""CCC""","""2014B"""
24290,169380,242,null,null,"""CCC""","""2014B"""
24290,555008,266,null,null,"""CCC""","""2014B"""
24299,555498,285,null,null,"""CCC""","""2014J"""
25368,2341830,279,null,null,"""DDD""","""2014J"""
…,…,…,…,…,…,…
34866,404170,592,222,370,"""FFF""","""2013B"""
34866,549792,592,222,370,"""FFF""","""2013B"""
34865,549792,591,222,369,"""FFF""","""2013B"""


说明这 85 rows 其实混合了两种情况：

date = null 的 rows 是 Exam，所以无法计算 submission_delay

其他 rows 的 submission_delay 非常大，例如 364–370 days，明显不是普通 late submission

所以这些 records 更像特殊 administrative / delayed recording，而不是正常 assessment timing。

In [61]:
# 检查这 85 rows 分别属于哪些 assessment_type

(
    submitted_date_check
    .filter(
        pl.col("date_submitted") >
        pl.col("module_presentation_length")
    )
    .join(
        assessments.select(
            ["id_assessment", "assessment_type"]
        ),
        on="id_assessment",
        how="left"
    )
    .group_by("assessment_type")
    .len()
    .sort("len", descending=True)
)

assessment_type,len
str,u32
"""CMA""",70
"""Exam""",12
"""TMA""",3


大部分异常集中在 CMA，而且很多来自 Withdrawn students。现在最稳妥的 cleaning decision 仍然是：

Keep these rows in the cleaned data +
Do not overwrite date_submitted +
Flag them as unusual +
Exclude them automatically when building cutoff-based early features

In [63]:
# 建立一个 indicator，标记 date_submitted 是否超过 module length

submitted_date_check = submitted_date_check.with_columns(
    (
        pl.col("date_submitted") >
        pl.col("module_presentation_length")
    )
    .cast(pl.Int8)
    .alias("submitted_after_module_end")
)

submitted_date_check.group_by(
    "submitted_after_module_end"
).len().sort(
    "submitted_after_module_end"
)

submitted_after_module_end,len
i8,u32
0,173827
1,85


indicator 数量完全正确：

submitted_after_module_end = 0: 173,827 rows

submitted_after_module_end = 1: 85 rows

这 85 rows 先保留，后面做 early prediction features 时按 cutoff filter，它们自然不会进入早期窗口。

In [64]:
# 检查 date_submitted 是否早于一个明显不合理的范围。先看最小值和 negative count：

student_assessment.select(
    pl.col("date_submitted").min().alias("min_date_submitted"),
    pl.col("date_submitted").max().alias("max_date_submitted"),
    (pl.col("date_submitted") < 0).sum().alias("negative_date_count")
)

min_date_submitted,max_date_submitted,negative_date_count
i64,i64,u32
-11,608,2057


结果说明：

date_submitted 最小是 -11 + 
最大是 608 + 
一共有 2057 个 negative submission dates

结合 OULAD description，negative date_submitted 是允许的，表示 submission occurred before the module presentation starte

所以这 2057 rows 不应删除或改成 0。

In [65]:
# 检查这些 negative rows 的 is_banked 分布

(
    student_assessment
    .filter(pl.col("date_submitted") < 0)
    .group_by("is_banked")
    .len()
    .sort("is_banked")
)

is_banked,len
i64,u32
0,148
1,1909


绝大多数 negative date_submitted 都是 banked assessments，这完全符合 OULAD 的 data structur

另外 148 条 non-banked negative submissions 也可能是正式开课前提前提交，仍然不应视为 error

当前 decision：

Keep all negative date_submitted values + 
Do not replace with 0 + 
Do not remove these rows

In [66]:
# 检查这 148 条 is_banked = 0 的最小和最大日期

student_assessment.filter(
    (pl.col("date_submitted") < 0) &
    (pl.col("is_banked") == 0)
).select(
    pl.col("date_submitted").min().alias("min_date"),
    pl.col("date_submitted").max().alias("max_date"),
    pl.len().alias("row_count")
)

min_date,max_date,row_count
i64,i64,u32
-11,-1,148


范围很合理：

non-banked negative submissions 只有 -11 到 -1

都发生在 course start 前 11 天以内

更像 legitimate early submissions，而不是异常日期

所以这 148 rows 也全部保留，不需要 indicator

## 检查 batch

In [68]:
# 检查 student_info.final_result 是否只有 description 中规定的四类

student_info.group_by(
    "final_result"
).len().sort(
    "final_result"
)

final_result,len
str,u32
"""Distinction""",3024
"""Fail""",7052
"""Pass""",12361
"""Withdrawn""",10156


#### categorical values

In [ ]:
# 只报告 missing values 和 unexpected categories，不再打印所有正常类别。

expected_categories = {
    "gender": {"F", "M"},
    "highest_education": {
        "No Formal quals",
        "Lower Than A Level",
        "A Level or Equivalent",
        "HE Qualification",
        "Post Graduate Qualification"
    },
    "imd_band": {
        "0-10%",
        "10-20%",
        "20-30%",
        "30-40%",
        "40-50%",
        "50-60%",
        "60-70%",
        "70-80%",
        "80-90%",
        "90-100%",
        "Unknown"
    },
    "age_band": {
        "0-35",
        "35-55",
        "55<="
    },
    "disability": {"N", "Y"},
    "final_result": {
        "Distinction",
        "Pass",
        "Fail",
        "Withdrawn"
    },
    "assessment_type": {
        "TMA",
        "CMA",
        "Exam"
    }
}

checks = [
    ("student_info", student_info, [
        "gender",
        "highest_education",
        "imd_band",
        "age_band",
        "disability",
        "final_result"
    ]),
    ("assessments", assessments, [
        "assessment_type"
    ])
]

for dataset_name, df, columns in checks:
    print(f"\n{dataset_name.upper()}")

    for column in columns:
        observed = set(
            df.select(column)
            .drop_nulls()
            .unique()
            .get_column(column)
            .to_list()
        )

        unexpected = observed - expected_categories[column]
        missing = df.select(
            pl.col(column).null_count()
        ).item()

        print(
            f"{column:<22} "
            f"missing={missing:<5} "
            f"unexpected={sorted(unexpected)}"
        )


STUDENT_INFO
gender                 missing=0     unexpected=[]
highest_education      missing=0     unexpected=[]
imd_band               missing=0     unexpected=[]
age_band               missing=0     unexpected=[]
disability             missing=0     unexpected=[]
final_result           missing=0     unexpected=[]

ASSESSMENTS
assessment_type        missing=0     unexpected=[]


这个 batch check 全部通过：

所有 checked categorical variables 都没有 missing values

没有 unexpected categories

imd_band 的 "Unknown" 处理也正常

assessment_type 只有 TMA / CMA / Exam

#### numerical ranges

In [72]:
numerical_checks = {
    "courses": {
        "data": courses,
        "rules": {
            "module_presentation_length": (1, None)
        }
    },
    "assessments": {
        "data": assessments,
        "rules": {
            "weight": (0, 100)
        }
    },
    "student_info": {
        "data": student_info,
        "rules": {
            "num_of_prev_attempts": (0, None),
            "studied_credits": (0, None)
        }
    },
    "student_assessment": {
        "data": student_assessment,
        "rules": {
            "is_banked": (0, 1),
            "score": (0, 100)
        }
    }
}

for dataset_name, config in numerical_checks.items():
    df = config["data"]

    print(f"\n{dataset_name.upper()}")

    for column, (lower, upper) in config["rules"].items():
        condition = pl.lit(False)

        if lower is not None:
            condition = condition | (pl.col(column) < lower)

        if upper is not None:
            condition = condition | (pl.col(column) > upper)

        invalid_count = (
            df.filter(
                pl.col(column).is_not_null() & condition
            ).height
        )

        summary = df.select(
            pl.col(column).min().alias("min"),
            pl.col(column).max().alias("max"),
            pl.col(column).null_count().alias("missing")
        ).row(0)

        print(
            f"{column:<30} "
            f"min={summary[0]} "
            f"max={summary[1]} "
            f"missing={summary[2]} "
            f"invalid={invalid_count}"
        )


COURSES
module_presentation_length     min=234 max=269 missing=0 invalid=0

ASSESSMENTS
weight                         min=0.0 max=100.0 missing=0 invalid=0

STUDENT_INFO
num_of_prev_attempts           min=0 max=6 missing=0 invalid=0
studied_credits                min=30 max=655 missing=0 invalid=0

STUDENT_ASSESSMENT
is_banked                      min=0 max=1 missing=0 invalid=0
score                          min=0 max=100 missing=173 invalid=0


这个 numerical batch check 也通过了：

module_presentation_length: valid

weight: valid

num_of_prev_attempts: valid

studied_credits: valid

is_banked: valid

score: 所有 non-missing values 都在 0–100

唯一需要保留处理的是之前已经标记的 173 missing scores

#### missing-value summary batch check

In [ ]:
# 把 missing-value summary 合并成一个 compact table，就不会被截断

missing_rows = []

for dataset_name, df in datasets.items():
    for column in df.columns:
        missing_count = df.select(
            pl.col(column).null_count()
        ).item()

        if missing_count > 0:
            missing_rows.append({
                "dataset": dataset_name,
                "column": column,
                "missing_count": missing_count,
                "missing_percent": round(
                    missing_count / df.height * 100,
                    2
                )
            })

missing_summary = pl.DataFrame(missing_rows)

missing_summary

dataset,column,missing_count,missing_percent
str,str,i64,f64
"""assessments""","""date""",11,5.34
"""vle""","""week_from""",5243,82.39
"""vle""","""week_to""",5243,82.39
"""student_registration""","""date_registration""",45,0.14
"""student_registration""","""date_unregistration""",22521,69.1
"""student_assessment""","""score""",173,0.1


说明所有 remaining missing values 都已经解释清楚了，不需要额外删除或强制 imputation：

assessments.date: 11 个，全部为 Exam → keep as null

vle.week_from/week_to: 82.39%，成对缺失 → keep as null

date_registration: 45 个 → keep as null + registration_date_missing

date_unregistration: 22,521 个，代表 no withdrawal → keep as null + has_unregistered

score: 173 个 → keep as null + score_missing

所以 missing-value cleaning 已完成。

#### referential integrity

In [75]:
# 确认不同 datasets 的 IDs 都能正确对应：

integrity_checks = []

# Every student assessment should match an assessment
unmatched_assessments = (
    student_assessment
    .join(
        assessments.select("id_assessment"),
        on="id_assessment",
        how="anti"
    )
    .height
)

integrity_checks.append({
    "check": "student_assessment → assessments",
    "unmatched_rows": unmatched_assessments
})

# Every student registration should match student_info
unmatched_registration = (
    student_registration
    .join(
        student_info.select(
            ["code_module", "code_presentation", "id_student"]
        ),
        on=["code_module", "code_presentation", "id_student"],
        how="anti"
    )
    .height
)

integrity_checks.append({
    "check": "student_registration → student_info",
    "unmatched_rows": unmatched_registration
})

# Every assessment should match a course presentation
unmatched_courses = (
    assessments
    .join(
        courses.select(
            ["code_module", "code_presentation"]
        ),
        on=["code_module", "code_presentation"],
        how="anti"
    )
    .height
)

integrity_checks.append({
    "check": "assessments → courses",
    "unmatched_rows": unmatched_courses
})

# Every VLE site should match a course presentation
unmatched_vle_courses = (
    vle
    .join(
        courses.select(
            ["code_module", "code_presentation"]
        ),
        on=["code_module", "code_presentation"],
        how="anti"
    )
    .height
)

integrity_checks.append({
    "check": "vle → courses",
    "unmatched_rows": unmatched_vle_courses
})

pl.DataFrame(integrity_checks)

check,unmatched_rows
str,i64
"""student_assessment → assessmen…",0
"""student_registration → student…",0
"""assessments → courses""",0
"""vle → courses""",0


全部都是 0，说明这几组 key relationships 都完整：

student_assessment 中的每个 id_assessment 都能在 assessments 找到

student_registration 中的每个 student-module-presentation record 都能在 student_info 找到

assessments 和 vle 中的每个 presentation 都能在 courses 找到

所以 referential integrity check 通过，暂时没有 orphan records，也不需要删除 rows

### 把 cleaning changes 整理回各 datasets

In [76]:
# 把 submitted_after_module_end 合并回 student_assessment

student_assessment = (
    submitted_date_check
    .select(
        [
            "id_assessment",
            "id_student",
            "date_submitted",
            "is_banked",
            "score",
            "score_missing",
            "submitted_after_module_end"
        ]
    )
)

student_assessment.head()

id_assessment,id_student,date_submitted,is_banked,score,score_missing,submitted_after_module_end
i64,i64,i64,i64,i64,i8,i8
1752,11391,18,0,78,0,0
1752,28400,22,0,70,0,0
1752,31604,17,0,72,0,0
1752,32885,26,0,69,0,0
1752,38053,19,0,79,0,0


student_assessment 现在已经包含两个 cleaning indicators：

score_missing

submitted_after_module_end

而且原始 columns 都保留了，没有丢数据

In [77]:
# 做一次 final validation，确认 row count 没有在 merge 过程中变化

student_assessment.shape

(173912, 7)

row count 没变，说明 join 和 indicator merge 都没有造成 duplication 或 data loss

In [78]:
# 把六个较小 datasets 保存为 cleaned Parquet

courses.write_parquet(
    CLEAN_DIR / "courses_clean.parquet"
)

assessments.write_parquet(
    CLEAN_DIR / "assessments_clean.parquet"
)

vle.write_parquet(
    CLEAN_DIR / "vle_clean.parquet"
)

student_info.write_parquet(
    CLEAN_DIR / "student_info_clean.parquet"
)

student_registration.write_parquet(
    CLEAN_DIR / "student_registration_clean.parquet"
)

student_assessment.write_parquet(
    CLEAN_DIR / "student_assessment_clean.parquet"
)

print("Six cleaned datasets saved successfully.")

Six cleaned datasets saved successfully.


In [79]:
for file in sorted(CLEAN_DIR.glob("*.parquet")):
    size_mb = file.stat().st_size / (1024 ** 2)
    print(f"{file.name:<40} {size_mb:>8.2f} MB")

assessments_clean.parquet                    0.00 MB
courses_clean.parquet                        0.00 MB
student_assessment_clean.parquet             0.52 MB
student_info_clean.parquet                   0.14 MB
student_registration_clean.parquet           0.12 MB
vle_clean.parquet                            0.02 MB


# 进入最大的 dataset